# RLHF Clinical Red-Teaming — CLI Driver
**Audrey Tjokro · Stephen Dong · Niki Karanikola**

This notebook is a **driver only**: no training logic, no hyperparameters, no model code lives here. It clones the repo at a pinned commit, installs deps, sets secrets, and shells out to `python -m src ...`.

Edit the **CONFIG** cell to choose method (`baseline` / `dpo` / `ppo`) and any overrides; everything else can be "Run all".

Per-run artifacts sync to `gs://results_043026/<method>/<run-uuid>/` at the end of the run.

## 1. Mount Drive (optional — used only as a local cache for HF + checkpoints)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone repo at a specific commit/branch
Pin a SHA for paper-grade reproducibility. Pass `BRANCH = 'main'` for HEAD.

In [2]:
REPO_URL = 'https://github.com/stephendongg/rlhf-clinical-redteaming.git'
BRANCH   = 'combined'
REPO_DIR = '/content/rlhf-clinical-redteaming'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())
%cd $REPO_DIR

f5d17e14faf21bd75cbfa5606c81302b0d01afaa
/content/rlhf-clinical-redteaming


In [3]:
!cd {REPO_DIR} && git pull origin combined


From https://github.com/stephendongg/rlhf-clinical-redteaming
 * branch            combined   -> FETCH_HEAD
Already up to date.


## 3. Install requirements

In [4]:
%pip install -q -r requirements.txt
%pip install -q -e .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requir

## 4. Secrets + GCS auth
Add `OPENAI_API_KEY` to Colab Secrets (left sidebar → key icon).

In [5]:
from google.colab import userdata, auth
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY').strip()

# GCS auth — uses your Google account; bucket gs://results_043026 must be in project rlhf-clinical-redteaming.
auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = 'rlhf-clinical-redteaming'

## 5. CONFIG — pick method, run name, any overrides

In [10]:
METHOD     = 'ppo'
CONFIG     = f'configs/{METHOD}.yaml'
RUN_NAME   = 'ppo-experiment-6'
GCS_BUCKET = 'gs://results_043026'

OVERRIDES = [
    'data.n_train=50',
    'data.n_dev=20',
    # 'dpo.n_per_seed=2',
    'judge_backend=openai',
    'judge_model=gpt-4o-mini',
    # Push utilization:
    'attacker_max_new_tokens=256',     # was 256 — longer generations
    'target_max_new_tokens=256',
    'lora.r=16',                       # was 16 — fatter adapter
    'lora.alpha=32',                   # keep alpha = 2*r
    'target_kl=0.3',
]

EXTRA_FLAGS = []

EXTRA_FLAGS = ['--allow-dirty']

In [11]:
import shlex
cmd = ['python', '-m', 'src', METHOD,
       '--config', CONFIG,
       '--run-name', RUN_NAME,
       '--gcs-bucket', GCS_BUCKET]
for o in OVERRIDES:
    cmd += ['--override', o]
cmd += EXTRA_FLAGS
print('$', ' '.join(shlex.quote(c) for c in cmd))

$ python -m src ppo --config configs/ppo.yaml --run-name ppo-experiment-4 --gcs-bucket gs://results_043026 --override data.n_train=50 --override data.n_dev=20 --override judge_backend=openai --override judge_model=gpt-4o-mini --override attacker_max_new_tokens=256 --override target_max_new_tokens=256 --override lora.r=16 --override lora.alpha=32 --override target_kl=1.0 --allow-dirty


## 6. Run the CLI (artifacts sync to GCS at the end automatically)

In [12]:
# import subprocess, sys
# result = subprocess.run(cmd)
# sys.exit(result.returncode) if result.returncode else print('OK')

In [13]:
import subprocess, sys, time
print("Starting run...")
print("$", " ".join(cmd))
print("=" * 80)
start = time.time()
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
elapsed = time.time() - start
print("=" * 80)
print(f"Finished with return code: {returncode}")
print(f"Elapsed time: {elapsed/60:.2f} minutes")
if returncode != 0:
    raise RuntimeError(f"CLI failed with return code {returncode}")
else:
    print("OK")


Starting run...
$ python -m src ppo --config configs/ppo.yaml --run-name ppo-experiment-4 --gcs-bucket gs://results_043026 --override data.n_train=50 --override data.n_dev=20 --override judge_backend=openai --override judge_model=gpt-4o-mini --override attacker_max_new_tokens=256 --override target_max_new_tokens=256 --override lora.r=16 --override lora.alpha=32 --override target_kl=1.0 --allow-dirty
13:25:29 INFO    redteam_rlhf.cli | Git SHA: f5d17e14faf21bd75cbfa5606c81302b0d01afaa (dirty)
13:25:29 INFO    redteam_rlhf.cli | GPU: NVIDIA A100-SXM4-40GB
13:25:29 INFO    redteam_rlhf.cli | Seeded RNGs with seed=42
13:25:29 INFO    redteam_rlhf.cli | Run ID: c46cbc6e9df1 | name=ppo-experiment-4
13:25:29 INFO    redteam_rlhf.results | Run dir: results/runs/c46cbc6e9df1
13:25:39 INFO    numexpr.utils | NumExpr defaulting to 12 threads.
13:25:42 INFO    datasets | TensorFlow version 2.20.0 available.
13:25:42 INFO    datasets | JAX version 0.7.2 available.
13:25:43 INFO    httpx | HTTP Requ

## 7. (Optional) Inspect this run's artifacts in GCS

In [ ]:
!gsutil ls -r {GCS_BUCKET}/{METHOD}/ | tail -30